# Tune LR (Nomogram), SVC, LightGBM, KNN, XGBoost

Import libraries/packages + preprocessed data

In [ ]:
import sys

sys.path.append("../")
from src.config import BASE_PATH
from src.data_utils import get_data
from src.tune_base import (
    lr_model_builder,
    lightgbm_model_builder,
    xgb_model_builder,
    svc_model_builder,
    knn_model_builder,
    tune_single_model,
    train_final_model,
)

Import data/ notebook globals

In [ ]:
file_dir = BASE_PATH / "data" / "processed"

DATA_DICT = {"base": get_data(is_nomo=False), "nomo": get_data(is_nomo=True)}

## Dont make max timeout below 15s
MODEL_CONFIG = {
    "lr": lr_model_builder,
    "lgbm": lightgbm_model_builder,
    "xgb": xgb_model_builder,
    "svc": svc_model_builder,
    "knn": knn_model_builder,
}

LOG_PATH = BASE_PATH / "models" / "logs"
RESULT_PATH = BASE_PATH / "models" / "tune_results"
NUM_TRIALS = 3
NUM_PARALLEL_TRIALS = 1

## TUNE

Single Run

In [ ]:
# model_abrv = "svc"
# tune_single_model(
#     model_builder=MODEL_CONFIG[model_abrv],
#     model_name=model_abrv,
#     X_train = DATA_DICT['base']['X_train'],
#     y_train = DATA_DICT['base']['y_train'],
#     scoring="roc_auc",
#     log_file_path=LOG_PATH / f"{model_abrv}.log",
#     save_path=RESULT_PATH / f"{model_abrv}.json",
#     n_trials=NUM_TRIALS,
#     clear_progress=True,
# )

Sequential

In [ ]:
for model_name, model_builder in MODEL_CONFIG.items():
    print(f"Working on: {model_name}...")
    data_type = "nomo" if model_name == "lr" else "base"
    tune_single_model(
        model_builder=model_builder,
        model_name=model_name,
        X_train=DATA_DICT[data_type]["X_train"],
        y_train=DATA_DICT[data_type]["y_train"],
        scoring="roc_auc",
        log_file_path=LOG_PATH / f"{model_name}.log",
        save_path=RESULT_PATH / f"{model_name}.json",
        n_trials=NUM_TRIALS,
        clear_progress=True,
    )

## TRAIN MODELS

In [ ]:
for model_name, model_builder in MODEL_CONFIG.items():
    print(f"Working on: {model_name}...")
    data_type = "nomo" if model_name == "lr" else "base"
    train_final_model(
        results_path=RESULT_PATH / f"{model_name}.json",
        model_builder=model_builder,
        model_name=model_name,
        X_train=DATA_DICT[data_type]["X_train"],
        y_train=DATA_DICT[data_type]["y_train"],
        X_test=None,
        y_test=None,
        model_save_path=BASE_PATH / "models" / "trained" / f"{model_name}.joblib",
    )